# Black-box Question Generation Pipeline

This notebook generates domain-adapted English questions using predefined templates and entity slots.
It is designed to be **non-iterative** (one-shot generation) and suitable for testing across different domain descriptions.

Notebook structure:
1. Step 1 — Basic setup (templates & domain weights)
2. Step 2 — Input database description
3. Step 3 — Allocate templates according to domain
4. Step 4 — Generate English questions (slot filling)
5. Step 5 — Export results

## 1. Step 1 — Basic setup (templates & domain weights)

In [36]:
import random
import json

# English template library (A-F categories)
TEMPLATES = {
    "A": [
        "What is [ENTITY]?",
        "What are the main features of [ENTITY]?",
        "Can you give an example of [ENTITY]?",
        "How does [ENTITY A] differ from [ENTITY B]?"
    ],
    "B": [
        "What are the key steps in [PROCESS]?",
        "How is [TASK] performed?",
        "Which methods are usually used for [TASK]?"
    ],
    "C": [
        "Why does [PHENOMENON] occur?",
        "What are the causes of [PHENOMENON]?",
        "What evidence supports [CLAIM]?"
    ],
    "D": [
        "How has [ENTITY] changed over time?",
        "What are the major historical milestones of [ENTITY]?",
        "What trends can be observed in [ENTITY]?"
    ],
    "E": [
        "What is the practical application of [ENTITY]?",
        "How does [ENTITY] relate to real-world problems?",
        "What impact might [ENTITY] have on society?"
    ],
    "F": [
        "What are the main controversies about [ENTITY] in academia?",
        "How do different viewpoints on [ENTITY] conflict?",
        "What gaps currently exist in research on [ENTITY]?"
    ]
}

# Domain weights mapping (A-F categories)
DOMAIN_WEIGHTS = {
    "General Knowledge":  {"A":0.30,"B":0.15,"C":0.15,"D":0.15,"E":0.15,"F":0.10},
    "Academic/Research":   {"A":0.15,"B":0.25,"C":0.15,"D":0.20,"E":0.15,"F":0.10},
    "Medical/Clinical":    {"A":0.15,"B":0.30,"C":0.25,"D":0.10,"E":0.15,"F":0.05},
    "Legal/Regulations":   {"A":0.10,"B":0.20,"C":0.30,"D":0.20,"E":0.15,"F":0.05},
    "News/Current Events": {"A":0.20,"B":0.15,"C":0.20,"D":0.15,"E":0.20,"F":0.10},
    "Social Media/Chat":   {"A":0.15,"B":0.15,"C":0.15,"D":0.10,"E":0.30,"F":0.15},
    "Technical Docs/FAQ":  {"A":0.25,"B":0.25,"C":0.30,"D":0.10,"E":0.05,"F":0.05},
    "Historical Archives": {"A":0.20,"B":0.20,"C":0.15,"D":0.25,"E":0.10,"F":0.10},
    "Finance":             {"A":0.25,"B":0.25,"C":0.15,"D":0.10,"E":0.15,"F":0.10} 
}

## 2. Step 2 — Input database description

In [37]:
# database_desc = {
#     "name": "chatdoctor",
#     "type": "Medical/Clinical",  # 映射英文类别
#     "intro": "Real conversations between patients and doctors from multiple medical dialogue websites, containing a wealth of authentic case records covering disease symptoms, diagnoses, and treatment recommendations."
# }

database_desc = {
    "name": "fiqa",
    "type": "Finance",  # 映射英文类别
    "intro": "a financial sentiment analysis benchmark derived from real-world sources such as StockTwits posts and financial news headlines.it enables models to understand market sentiment and investor opinions in financial contexts."
}

print("Database description:")
for k, v in database_desc.items():
    print(f"{k}: {v}")

Database description:
name: fiqa
type: Finance
intro: a financial sentiment analysis benchmark derived from real-world sources such as StockTwits posts and financial news headlines.it enables models to understand market sentiment and investor opinions in financial contexts.


## 3. Step 3 — Allocate templates according to domain

In [38]:
# Allocate templates according to domain

def allocate_templates(domain_type, total_questions=50):
    # 根据 domain_type 从 DOMAIN_WEIGHTS 获取比例，并计算每类模板数量
    if domain_type not in DOMAIN_WEIGHTS:
        raise ValueError(f"Domain type '{domain_type}' not found in DOMAIN_WEIGHTS")
    weights = DOMAIN_WEIGHTS[domain_type]
    allocation = {k: int(v * total_questions) for k, v in weights.items()}
    return allocation

# 分配模板
allocation = allocate_templates(database_desc["type"], total_questions=500)
print("Template allocation (number of questions per category):")
print(allocation)

Template allocation (number of questions per category):
{'A': 125, 'B': 125, 'C': 75, 'D': 50, 'E': 75, 'F': 50}


## 4. Step 4 — Generate English questions (slot filling)

In [41]:
### get entity pool
from openai import OpenAI

# 初始化 LLM（示例为 OpenAI API）
client = OpenAI(base_url="http://localhost:22999/v1", api_key="EMPTY")

def generate_entity_pool_llm(db_description, num_entities=30):
    prompt = f"""
    Given the following database description:
    \"\"\"{db_description['intro']}\"\"\"

    Please generate a list of {num_entities} relevant entities in English, without any extra explanation, prefix and suffix. Output as a JSON array.
    """

    response = client.chat.completions.create(
        model="./Models/Qwen3-32B",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        top_p=0.8,
        max_tokens=2048
    )
    # 获取 content
    content = getattr(response.choices[0].message, "content", "")
    content = content.strip() if content else ""

    return content

# 生成实体池
content = generate_entity_pool_llm(database_desc, num_entities=100)
print("Generated entity pool:", content)

Generated entity pool: [
  "AAPL",
  "MSFT",
  "GOOGL",
  "AMZN",
  "TSLA",
  "FB",
  "NVDA",
  "BABA",
  "CSCO",
  "INTC",
  "S&P 500",
  "Dow Jones",
  "Nikkei 225",
  "FTSE 100",
  "DAX",
  "NASDAQ",
  "Russell 2000",
  "Gold",
  "Silver",
  "Oil",
  "Bitcoin",
  "Ethereum",
  "Federal Reserve",
  "SEC",
  "ECB",
  "IMF",
  "World Bank",
  "GDP",
  "CPI",
  "Unemployment Rate",
  "Interest Rates",
  "Quantitative Easing",
  "Inflation",
  "Deflation",
  "Bull Market",
  "Bear Market",
  "Market Crash",
  "Correction",
  "Volatility",
  "Short Squeeze",
  "Long Position",
  "Short Position",
  "Hedge Fund",
  "Penny Stock",
  "Blue Chip",
  "ETF",
  "Mutual Fund",
  "Options",
  "Futures",
  "Derivatives",
  "Leverage",
  "Liquidity",
  "Dividend Yield",
  "Earnings Per Share",
  "Revenue Growth",
  "Profit Margin",
  "P/E Ratio",
  "Debt-to-Equity",
  "ROE",
  "ROA",
  "Market Cap",
  "IPO",
  "M&A",
  "Acquisition",
  "Merger",
  "Bankruptcy",
  "Delisting",
  "Short Interest",
  "

In [42]:
entity_pool = [
  "AAPL",
  "MSFT",
  "GOOGL",
  "AMZN",
  "TSLA",
  "FB",
  "NVDA",
  "BABA",
  "CSCO",
  "INTC",
  "S&P 500",
  "Dow Jones",
  "Nikkei 225",
  "FTSE 100",
  "DAX",
  "NASDAQ",
  "Russell 2000",
  "Gold",
  "Silver",
  "Oil",
  "Bitcoin",
  "Ethereum",
  "Federal Reserve",
  "SEC",
  "ECB",
  "IMF",
  "World Bank",
  "GDP",
  "CPI",
  "Unemployment Rate",
  "Interest Rates",
  "Quantitative Easing",
  "Inflation",
  "Deflation",
  "Bull Market",
  "Bear Market",
  "Market Crash",
  "Correction",
  "Volatility",
  "Short Squeeze",
  "Long Position",
  "Short Position",
  "Hedge Fund",
  "Penny Stock",
  "Blue Chip",
  "ETF",
  "Mutual Fund",
  "Options",
  "Futures",
  "Derivatives",
  "Leverage",
  "Liquidity",
  "Dividend Yield",
  "Earnings Per Share",
  "Revenue Growth",
  "Profit Margin",
  "P/E Ratio",
  "Debt-to-Equity",
  "ROE",
  "ROA",
  "Market Cap",
  "IPO",
  "M&A",
  "Acquisition",
  "Merger",
  "Bankruptcy",
  "Delisting",
  "Short Interest",
  "Volume",
  "Open Interest",
  "Put/Call Ratio",
  "VIX",
  "Bond Yields",
  "Yield Curve",
  "Treasury Bonds",
  "Corporate Bonds",
  "Mortgage Rates",
  "Trade War",
  "Tariffs",
  "Sanctions",
  "Pandemic",
  "Geopolitical Risk",
  "Central Bank Policy",
  "Monetary Policy",
  "Fiscal Policy",
  "Tax Cuts",
  "Government Spending",
  "Recession",
  "Recovery",
  "Growth",
  "Stagflation",
  "Hyperinflation",
  "Currency Depreciation",
  "Currency Appreciation",
  "FX Rates",
  "USD",
  "EUR",
  "GBP",
  "JPY",
  "CHF",
  "Emerging Markets",
  "Developed Markets",
  "Market Sentiment",
  "Investor Confidence",
  "Panic Selling",
  "FOMO",
  "Fear of Missing Out",
  "Herd Mentality",
  "Algorithmic Trading",
  "High-Frequency Trading",
  "Retail Investors",
  "Institutional Investors",
  "Day Trading",
  "Swing Trading",
  "Position Trading",
  "Technical Analysis",
  "Fundamental Analysis",
  "Sentiment Analysis",
  "News Headlines",
  "Social Media",
  "StockTwits",
  "Reddit",
  "Twitter",
  "Bullish",
  "Bearish",
  "Neutral",
  "Outperform",
  "Underperform",
  "Market Outperform",
  "Sector Rotation",
  "Dividend Growth",
  "Share Buybacks",
  "Stock Splits",
  "Reverse Splits",
  "Insider Trading",
  "Earnings Surprise",
  "Guidance",
  "Revenue Miss",
  "Earnings Beat",
  "Product Launch",
  "Regulatory Approval",
  "Product Recall",
  "Management Change",
  "Legal Issues",
  "Lawsuit",
  "Environmental Concerns",
  "Corporate Governance",
  "ESG",
  "Sustainability",
  "Climate Risk",
  "Renewable Energy",
  "Green Bonds",
  "Carbon Emissions",
  "Supply Chain Disruption",
  "Labor Strikes",
  "Natural Disasters",
  "Cybersecurity Breach",
  "Data Privacy",
  "AI Adoption",
  "Blockchain",
  "Fintech",
  "DeFi",
  "NFTs",
  "Metaverse",
  "5G",
  "Electric Vehicles",
  "Clean Energy",
  "Semiconductors",
  "Biotechnology",
  "Pharmaceuticals",
  "Healthcare",
  "Consumer Discretionary",
  "Consumer Staples",
  "Technology",
  "Communication Services",
  "Industrials",
  "Energy",
  "Materials",
  "Utilities",
  "Real Estate",
  "Financials",
  "Healthcare Providers",
  "Education Sector",
  "Travel & Leisure",
  "Retail",
  "Aerospace & Defense"
]

entity_pool = list(set([e.replace('"','') for e in entity_pool]))

In [43]:
len(entity_pool)

186

In [44]:
# Generate English Questions with Multiple Entities
def generate_questions_multi_entity(allocation, entity_pool, variants_per_template=2):
    """
    allocation: dict, 模板类别 -> 生成问题数量
    entity_pool: list of str, 可用实体
    variants_per_template: 每个模板生成多少变体
    """
    questions = []

    for cat, num in allocation.items():
        templates = TEMPLATES[cat]
        for _ in range(num):
            tmpl = random.choice(templates)
            for _ in range(variants_per_template):
                # 从实体池随机选择实体
                entity_main = random.choice(entity_pool)
                entity_a = random.choice(entity_pool)
                entity_b = random.choice(entity_pool)
                process_task = random.choice(entity_pool)
                phenomenon = random.choice(entity_pool)
                claim = random.choice(entity_pool)
                
                # 填充模板槽位
                q = tmpl.replace("[ENTITY]", entity_main)
                q = q.replace("[ENTITY A]", entity_a)
                q = q.replace("[ENTITY B]", entity_b)
                q = q.replace("[PROCESS]", process_task)
                q = q.replace("[TASK]", process_task)
                q = q.replace("[PHENOMENON]", phenomenon)
                q = q.replace("[CLAIM]", claim)
                
                questions.append(q)
    return questions

# 多实体生成
# entity_pool = ["diabetes", "hypertension", "asthma", "cancer", "flu", "COVID-19", "migraine"]
questions_multi = generate_questions_multi_entity(allocation, entity_pool, variants_per_template=3)

# 查看前10个示例
print("Sample multi-entity questions:")
for q in questions_multi[:10]:
    print("-", q)

Sample multi-entity questions:
- Can you give an example of Lawsuit?
- Can you give an example of Market Outperform?
- Can you give an example of Healthcare?
- How does USD differ from Pharmaceuticals?
- How does Twitter differ from Recession?
- How does Geopolitical Risk differ from IPO?
- What are the main features of GBP?
- What are the main features of Technical Analysis?
- What are the main features of Sustainability?
- What are the main features of Investor Confidence?


In [45]:
questions_multi

['Can you give an example of Lawsuit?',
 'Can you give an example of Market Outperform?',
 'Can you give an example of Healthcare?',
 'How does USD differ from Pharmaceuticals?',
 'How does Twitter differ from Recession?',
 'How does Geopolitical Risk differ from IPO?',
 'What are the main features of GBP?',
 'What are the main features of Technical Analysis?',
 'What are the main features of Sustainability?',
 'What are the main features of Investor Confidence?',
 'What are the main features of Bearish?',
 'What are the main features of Share Buybacks?',
 'Can you give an example of Derivatives?',
 'Can you give an example of Liquidity?',
 'Can you give an example of Bitcoin?',
 'Can you give an example of Sentiment Analysis?',
 'Can you give an example of Central Bank Policy?',
 'Can you give an example of NFTs?',
 'What is VIX?',
 'What is Neutral?',
 'What is Materials?',
 'Can you give an example of Data Privacy?',
 'Can you give an example of Investor Confidence?',
 'Can you give

In [46]:
len(questions_multi)

1500

## 5. Step 5 — Export results

In [47]:
questions_multi = random.sample(questions_multi, 500)

In [48]:
# 保存为 JSON Lines
output_file = "questions.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for idx, q in enumerate(questions_multi, start=1):
        entry = {"_id": str(idx), "text": q, "metadata": {}}
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"{len(questions_multi)} questions saved to {output_file}")

500 questions saved to questions.jsonl
